In [ ]:
from pathlib import Path

ROOT = Path("./results")    # 根目录：包含 gcd/ 和 openset/
SETTING = "openset"             # 在 "gcd" 与 "openset" 间切换

# 每个 setting 对应要读取的任务名；留空 [] 或 ["*"] 表示扫描该 setting 下所有含 results.csv 的任务
TASKS_MAP = {
    "gcd":     ["alup", "deepaligned", "dpn", "geoid", "loop", "plm_gcd", "sdc", "tan"],
    "openset": ["ab", "adb", "clap", "deepunk", "doc", "dyen", "knncon", "plm_ood", "scl"],
}

# 每个 setting 对应要展示的指标（按截图风格）
METRICS_MAP = {
    "gcd":     ["K-ACC"],
    "openset": ["ACC"],
}

ROUND_DECIMALS = 2          # 数值保留小数位
OUTPUT_TEX = f"{SETTING}_wide_table.tex"  # 输出的 LaTeX 文件名
HIGHLIGHT_TOP2 = True       # 是否加粗最佳、下划线次佳

In [14]:
# ========== 仅替换你“从读取到生成 LaTeX”的这部分 ==========

import pandas as pd
import numpy as np

base = ROOT / SETTING
metrics = METRICS_MAP[SETTING]

# 可选：自定义显示顺序（不需要可置为 None）
KCR_ORDER = [0.25, 0.50, 0.75]   # 你的数据常见顺序
LCR_ORDER = [0.1, 0.5, 1.0]      # labeled_ratio 的三种

# 1) 任务列表
task_list = TASKS_MAP.get(SETTING, [])
if (not task_list) or (task_list == ["*"]):
    task_list = sorted(
        p.name for p in base.iterdir()
        if p.is_dir() and (p / "results.csv").exists()
    )
if not task_list:
    raise FileNotFoundError(f"未发现任务目录。检查 {base} 或填充 TASKS_MAP['{SETTING}'].")

# 2) 读取并合并
frames, missing = [], []
for task in task_list:
    csv_path = base / task / "results.csv"
    if not csv_path.exists():
        missing.append(str(csv_path)); continue
    df = pd.read_csv(csv_path)
    df["Method"] = df["method"] if "method" in df.columns else task.upper()
    df["KCR"] = df.get("known_cls_ratio", np.nan)
    df["LCR"] = df.get("labeled_ratio", np.nan)   # 新增：LCR
    frames.append(df)

if not frames:
    raise FileNotFoundError("没有可读的 results.csv：\n" + "\n".join(missing))

df = pd.concat(frames, ignore_index=True)

# 3) 只保留所需字段（新增 LCR）
required_cols = ["dataset", "Method", "KCR", "LCR"] + metrics
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"以下列在数据中缺失：{missing_cols}")

# 4) 聚合：对同一 (dataset, Method, KCR, LCR) 求均值（跨 seed/fold）
grouped = (
    df[required_cols]
    .groupby(["dataset", "Method", "KCR", "LCR"], as_index=False)
    .mean(numeric_only=True)
)

# 5) 透视：列为 (dataset, metric)，行索引 (KCR, LCR, Method)
long = grouped.melt(id_vars=["dataset", "Method", "KCR", "LCR"],
                    value_vars=metrics, var_name="metric", value_name="value")
pivot = long.pivot_table(index=["KCR", "LCR", "Method"],
                         columns=["dataset", "metric"],
                         values="value", aggfunc="mean")

# 6) 行列排序
# 行：先按 KCR，再按 LCR，再按 Method
pivot = pivot.sort_index(level=[0, 1, 2])

# 如果提供了自定义顺序，则按顺序重排（防止 0.1/0.5/1.0 无序）
if KCR_ORDER is not None or LCR_ORDER is not None:
    idx = pivot.index
    kcr_cat = pd.Categorical(idx.get_level_values(0), categories=KCR_ORDER or sorted(idx.get_level_values(0).unique()), ordered=True)
    lcr_cat = pd.Categorical(idx.get_level_values(1), categories=LCR_ORDER or sorted(idx.get_level_values(1).unique()), ordered=True)
    method = idx.get_level_values(2)
    new_index = pd.MultiIndex.from_arrays([kcr_cat, lcr_cat, method], names=idx.names)
    pivot = pivot.reindex(new_index).sort_index(level=[0,1,2])

# 列：对每个 dataset 保持 metrics 的既定顺序
datasets_order = sorted(pivot.columns.get_level_values(0).unique().tolist())
metrics_order = metrics
existing_cols = set(map(tuple, pivot.columns.tolist()))
ordered_cols = [(ds, m) for ds in datasets_order for m in metrics_order if (ds, m) in existing_cols]
pivot = pivot.reindex(columns=pd.MultiIndex.from_tuples(ordered_cols))

# 7) 数值保留小数
pivot = pivot.round(ROUND_DECIMALS)

# 8) 高亮函数：按 (KCR, LCR) 分组，而不是只按 KCR
def format_with_highlight(df_block):
    """
    对一个 (dataset, metric) 列块，按 (KCR, LCR) 分组，高亮 top-1/2。
    """
    if not HIGHLIGHT_TOP2:
        return df_block.applymap(lambda x: f"{x:.{ROUND_DECIMALS}f}" if pd.notnull(x) else "")
    formatted = df_block.copy().astype(float)
    out = df_block.copy().astype(object)
    # level=0 -> KCR, level=1 -> LCR
    for (kcr, lcr), sub in formatted.groupby(level=[0,1]):
        vals = sub.iloc[:, 0]
        order = vals.sort_values(ascending=False, kind="mergesort")
        # 第一名加粗
        if len(order) >= 1 and pd.notnull(order.iloc[0]):
            top_idx = (kcr, lcr, order.index.get_level_values(2)[0])  # level=2 是 Method
            out.loc[top_idx, out.columns[0]] = "\\textbf{" + f"{order.iloc[0]:.{ROUND_DECIMALS}f}" + "}"
        # 第二名下划线
        if len(order) >= 2 and pd.notnull(order.iloc[1]):
            sec_idx = (kcr, lcr, order.index.get_level_values(2)[1])
            out.loc[sec_idx, out.columns[0]] = "\\underline{" + f"{order.iloc[1]:.{ROUND_DECIMALS}f}" + "}"
        # 其余正常格式
        for idx, v in sub.iloc[:, 0].items():
            if pd.isnull(v):
                out.loc[idx, out.columns[0]] = ""
            elif out.loc[idx, out.columns[0]] is sub.loc[idx, sub.columns[0]]:
                out.loc[idx, out.columns[0]] = f"{v:.{ROUND_DECIMALS}f}"
    return out

# 逐 dataset × metric 格式化
formatted_blocks = []
for ds in datasets_order:
    cols = []
    for m in metrics_order:
        if (ds, m) not in pivot.columns:
            continue
        col = pivot.loc[:, (ds, m)].to_frame()
        cols.append(format_with_highlight(col))
    if cols:
        ds_block = pd.concat(cols, axis=1)
        ds_block.columns = pd.MultiIndex.from_product([[ds], [c for c in metrics_order if (ds, c) in pivot.columns]])
        formatted_blocks.append(ds_block)

formatted = pd.concat(formatted_blocks, axis=1).fillna("")

# 9) 表头与行：左侧三列 -> KCR, LCR, Method
unique_ds = []
for d in formatted.columns.get_level_values(0):
    if d not in unique_ds:
        unique_ds.append(d)

head_line1 = " & ".join(
    ["KCR", "LCR", "Method"] +
    [f"\\multicolumn{{{len([m for m in metrics_order if (ds, m) in formatted.columns])}}}{{c}}{{{ds}}}" for ds in unique_ds]
) + " \\\\"

head_line2 = " & ".join(
    ["", "", ""] + [m for ds in unique_ds for m in metrics_order if (ds, m) in formatted.columns]
) + " \\\\ \\midrule"

def row_to_latex(idx, row):
    cells = [
        f"{idx[0]:.2f}" if pd.notnull(idx[0]) else "",         # KCR
        f"{idx[1]:.2f}" if pd.notnull(idx[1]) else "",         # LCR
        str(idx[2])                                            # Method
    ]
    for ds in unique_ds:
        for m in metrics_order:
            if (ds, m) in row.index:
                val = row[(ds, m)]
                cells.append(val if isinstance(val, str) else ("" if pd.isnull(val) else f"{val:.{ROUND_DECIMALS}f}"))
    return " & ".join(cells) + " \\\\"

body_lines, last_pair = [], (None, None)
for idx, row in formatted.iterrows():
    pair = (idx[0], idx[1])  # (KCR, LCR)
    if last_pair != (None, None) and pair != last_pair:
        body_lines.append("\\addlinespace")  # KCR×LCR 分块之间加间距
    body_lines.append(row_to_latex(idx, row))
    last_pair = pair

# 10) LaTeX（双栏全宽）
env = "table*"
size_cmd = "\\small"
tabcol = 4
col_count = 3 + sum(len([m for m in metrics_order if (ds, m) in formatted.columns]) for ds in unique_ds)

latex_table = "\n".join([
    f"\\begin{{{env}}}[t]",
    "\\centering",
    size_cmd,
    f"\\setlength{{\\tabcolsep}}{{{tabcol}pt}}",
    "\\resizebox{\\textwidth}{!}{%",
    "\\begin{tabular}{lll" + "r" * (col_count - 3) + "}",
    "\\toprule",
    head_line1,
    head_line2,
    "\n".join(body_lines),
    "\\bottomrule",
    "\\end{tabular}}",
    f"\\end{{{env}}}"
])

Path(OUTPUT_TEX).write_text(latex_table, encoding="utf-8")
print(f"[OK] 生成完成：{Path(OUTPUT_TEX).resolve()}")
print(latex_table[:800] + ("\\n... (truncated)" if len(latex_table) > 800 else ""))
# ========== 替换结束 ==========


[OK] 生成完成：/ssd/code/bolt/gcd_wide_table.tex
\begin{table*}[t]
\centering
\small
\setlength{\tabcolsep}{4pt}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lllrrrrrrrrrrrr}
\toprule
KCR & LCR & Method & \multicolumn{1}{c}{20NG} & \multicolumn{1}{c}{DBPedia} & \multicolumn{1}{c}{TREC} & \multicolumn{1}{c}{Yahoo} & \multicolumn{1}{c}{banking} & \multicolumn{1}{c}{clinc} & \multicolumn{1}{c}{ecdt} & \multicolumn{1}{c}{hwu} & \multicolumn{1}{c}{mcid} & \multicolumn{1}{c}{news} & \multicolumn{1}{c}{stackoverflow} & \multicolumn{1}{c}{thucnews} \\
 &  &  & K-ACC & K-ACC & K-ACC & K-ACC & K-ACC & K-ACC & K-ACC & K-ACC & K-ACC & K-ACC & K-ACC & K-ACC \\ \midrule
0.25 & 0.10 & ALUP & \textbf{98.94} & 82.22 &  & \textbf{72.38} & \underline{77.22} & \underline{91.66} & \underline{69.28} & \underline{79.39} & \textbf{76.71} & \underline{69.66} & \u\n... (truncated)


/tmp/ipykernel_2889949/3800342850.py:93: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (kcr, lcr), sub in formatted.groupby(level=[0,1]):
/tmp/ipykernel_2889949/3800342850.py:93: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (kcr, lcr), sub in formatted.groupby(level=[0,1]):
/tmp/ipykernel_2889949/3800342850.py:93: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (kcr, lcr), sub in formatted.groupby(level=[0,1])